# Splice impact check v1.1

Compare the original and donor-spliced station CSVs using the current index  
file and the **same calculation cells from `data_table_builder_v1.1.ipynb`**.  
Inputs stay live: this notebook reads the files present when it runs. Both  
passes use today's date from the start of this run, so a midnight boundary  
cannot give the two scenarios different calendars.  

Copy this notebook into the project's `scripts` folder before running it.  
It does not run the builder's export or plotting cells, refresh downloads,  
or modify input files. Optional reports go to `outputs/phase_one/splice_impact`.  

The original scenario has an empty replacement audit; the spliced scenario  
uses the saved audit. Published index values, seasonal labels, quality rules,  
completeness, units, sample SD and the nine-station ensemble rule stay the same.  

In [1]:
#dependencies
import ast
import json
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
#locate the project from either its root or its scripts directory
project_dir = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "scripts/data_table_builder_v1.1.ipynb").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Run this notebook from the project or its scripts folder.")

input_paths = {
    "original": project_dir / "station_data/ghcnd/all_station_data.csv",
    "spliced": project_dir / "station_data/ghcnd/record_splices/all_station_data_spliced.csv",
    "indices": project_dir / "indicies_data/phase_one/enso_historical_indicies.csv",
    "audit": project_dir / "station_data/ghcnd/record_splices/station_splice_audit.csv",
    "builder": project_dir / "scripts/data_table_builder_v1.1.ipynb",
}
for path in input_paths.values():
    if not path.is_file():
        raise FileNotFoundError(path)

comparison_date = pd.Timestamp.now(tz="America/Chicago").normalize().tz_localize(None)
save_comparison = True
comparison_dir = project_dir / "outputs/phase_one/splice_impact"

In [ ]:
#record what was actually read, without freezing or copying the inputs
def input_inventory():
    return pd.DataFrame([
        {
            "input": role,
            "file": path.relative_to(project_dir).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256(path.read_bytes()).hexdigest(),
            "modified_utc": pd.Timestamp(path.stat().st_mtime, unit="s", tz="UTC").isoformat(),
        }
        for role, path in input_paths.items()
    ])

inputs_before = input_inventory()
audit_records = pd.read_csv(
    input_paths["audit"],
    usecols=["time", "target_station", "donor_station", "variable", "reason"],
    dtype={"target_station": "string", "donor_station": "string",
           "variable": "string", "reason": "string"},
    parse_dates=["time"],
)
replacement_counts = (
    audit_records.groupby(["target_station", "donor_station", "variable", "reason"])
    .size().rename("replacements").reset_index()
)
display(replacement_counts)


# Run the existing builder calculations twice

The cell numbers below target the current v1.1 builder. Revisit this selection  
if its calculation cells are rearranged. Input paths, the scenario's audit and  
the shared comparison date are the only bindings redirected. The statistical  
formulas are read from the builder rather than maintained as a second copy.  


In [ ]:
#load calculation cells; exclude the writer, formatter and plotting cells
builder = json.loads(input_paths["builder"].read_text(encoding="utf-8"))
builder_cells = ["".join(cell["source"]) for cell in builder["cells"]
                 if cell["cell_type"] == "code"]
calculation_cells = [3, 4, 5, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
output_names = [
    "station_season_table", "station_summary", "ensemble_seasons",
    "ensemble_summary", "seasonal_index_table", "station_metadata",
]

def run_scenario(station_path, audit):
    namespace = {
        "pd": pd, "np": np, "Path": Path,
        "display": lambda *args, **kwargs: None,
        "comparison_station_path": station_path,
        "comparison_index_path": input_paths["indices"],
        "comparison_audit": audit.rename(columns={"target_station": "station"}).copy(),
        "comparison_date": comparison_date,
    }
    redirects = {
        "station_data": "comparison_station_path",
        "indicies_data": "comparison_index_path",
        "splice_audit_for_table": "comparison_audit",
        "analysis_today": "comparison_date",
    }
    redirected = set()
    for number in calculation_cells:
        tree = ast.parse(builder_cells[number - 1])
        for node in ast.walk(tree):
            if (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)
                    and node.func.attr in {"to_csv", "savefig", "mkdir"}):
                raise ValueError(f"Builder cell {number} now writes files; review calculation_cells.")
        for node in tree.body:
            if isinstance(node, ast.Assign) and len(node.targets) == 1:
                target = node.targets[0]
                if isinstance(target, ast.Name) and target.id in redirects:
                    node.value = ast.Name(id=redirects[target.id], ctx=ast.Load())
                    redirected.add(target.id)
        ast.fix_missing_locations(tree)
        exec(compile(tree, f"builder code cell {number}", "exec"), namespace)
    if redirected != set(redirects):
        raise ValueError("Builder inputs changed; review the comparison bindings.")
    result = {name: namespace[name].copy(deep=True) for name in output_names}
    namespace.clear()
    return result

In [ ]:
#no replacements apply in the original-input scenario
print("Original station records")
original_results = run_scenario(input_paths["original"], audit_records.iloc[:0])
print("Spliced station records")
spliced_results = run_scenario(input_paths["spliced"], audit_records)

pd.testing.assert_frame_equal(inputs_before, input_inventory())
pd.testing.assert_frame_equal(
    original_results["seasonal_index_table"], spliced_results["seasonal_index_table"]
)
pd.testing.assert_frame_equal(
    original_results["station_metadata"], spliced_results["station_metadata"]
)
print("Same inputs throughout the run; identical index classifications and primary stations.")


## Seasonal coverage and accepted weather values

A newly accepted season was missing or rejected before splicing and passes the  
existing completeness rule afterward. An unchanged accepted season appears in  
both scenarios. Ensemble coverage counts each season-year once, not once per  
index; ONI and RONI use the same weather eligibility.  

In [ ]:
metrics = [
    ("precip", "precip_in", "in"),
    ("snow", "snow_in", "in"),
    ("temp", "temp_f", "f"),
    ("max_monthly_snow", "max_monthly_snow_in", "in"),
    ("max_daily_tmax", "max_daily_tmax_f", "f"),
]
value_columns = [column for _, column, _ in metrics]

def seasonal_records(results):
    stations = results["station_season_table"][
        ["station", "season", "season_year"] + value_columns
    ].rename(columns={"station": "entity"}).copy()
    stations["scope"] = "station"
    yearly = results["ensemble_seasons"]
    oni = yearly.loc[yearly["index"].eq("ONI")].set_index(["season", "season_year"])
    roni = yearly.loc[yearly["index"].eq("RONI")].set_index(["season", "season_year"])
    pd.testing.assert_frame_equal(oni[value_columns], roni[value_columns])
    ensemble = roni[value_columns].reset_index()
    ensemble["entity"] = "nine_primary_stations"
    ensemble["scope"] = "ensemble"
    return pd.concat([stations, ensemble], ignore_index=True)

season_keys = ["scope", "entity", "season", "season_year"]
original_seasons = seasonal_records(original_results)
spliced_seasons = seasonal_records(spliced_results)
season_pairs = original_seasons.merge(
    spliced_seasons, on=season_keys, how="outer",
    suffixes=("_before", "_after"), validate="one_to_one", indicator=True,
)
if season_pairs["_merge"].ne("both").any():
    raise ValueError("The scenarios do not contain the same station/ensemble season keys.")

season_parts = []
for metric, column, unit in metrics:
    part = season_pairs[season_keys].copy()
    part["metric"], part["unit"] = metric, unit
    part["value_before"] = season_pairs[f"{column}_before"]
    part["value_after"] = season_pairs[f"{column}_after"]
    part["valid_before"] = part["value_before"].notna()
    part["valid_after"] = part["value_after"].notna()
    part["added"] = ~part["valid_before"] & part["valid_after"]
    part["lost"] = part["valid_before"] & ~part["valid_after"]
    part["changed_shared"] = (
        part["valid_before"] & part["valid_after"]
        & ~np.isclose(part["value_before"], part["value_after"], equal_nan=True)
    )
    season_parts.append(part)
seasonal_comparison = pd.concat(season_parts, ignore_index=True)
if seasonal_comparison["lost"].any() or seasonal_comparison["changed_shared"].any():
    raise ValueError("An originally accepted seasonal value changed; investigate the inputs.")
if seasonal_comparison.loc[
    ~seasonal_comparison["metric"].isin(["snow", "max_monthly_snow"]), "added"
].any():
    raise ValueError("Non-snow eligibility changed; inputs differ beyond the approved splices.")

coverage_keys = ["scope", "entity", "season", "metric", "unit"]
seasonal_splice_impact = seasonal_comparison.groupby(coverage_keys).agg(
    n_before=("valid_before", "sum"), n_after=("valid_after", "sum"),
    mean_before=("value_before", "mean"), mean_after=("value_after", "mean"),
    sd_before=("value_before", lambda values: values.std(ddof=1)),
    sd_after=("value_after", lambda values: values.std(ddof=1)),
    added=("added", "sum"), lost=("lost", "sum"),
    changed_shared=("changed_shared", "sum"),
).reset_index()
seasonal_splice_impact["mean_change"] = (
    seasonal_splice_impact["mean_after"] - seasonal_splice_impact["mean_before"]
)
seasonal_splice_impact["sd_change"] = (
    seasonal_splice_impact["sd_after"] - seasonal_splice_impact["sd_before"]
)
newly_accepted_seasons = seasonal_comparison.loc[seasonal_comparison["added"]].copy()
display(seasonal_splice_impact.loc[seasonal_splice_impact["added"].gt(0)])

## Changes to the actual ONI/RONI category summaries

These comparisons use the builder's station and ensemble summaries, including  
its mean, sample SD and metric-specific valid-season count. A mean or SD  
difference remains missing when it cannot be calculated in both scenarios;  
a previously unavailable value is not treated as zero. Category-mean changes  
can result from newly accepted years even when all previously accepted weather  
values remain unchanged.  


In [ ]:
def summary_comparison(before, after, keys):
    pairs = before.merge(after, on=keys, how="outer",
                         suffixes=("_before", "_after"), validate="one_to_one", indicator=True)
    if pairs["_merge"].ne("both").any():
        raise ValueError("The scenarios do not have identical classification groups.")
    parts = []
    for metric, _, unit in metrics:
        part = pairs[keys].copy()
        part["metric"], part["unit"] = metric, unit
        for field, source in [
            ("mean", f"{metric}_mean_{unit}"),
            ("sd", f"{metric}_sd_{unit}"),
            ("n", f"{metric}_n"),
        ]:
            part[f"{field}_before"] = pairs[f"{source}_before"]
            part[f"{field}_after"] = pairs[f"{source}_after"]
            part[f"{field}_change"] = part[f"{field}_after"] - part[f"{field}_before"]
        part["mean_newly_available"] = part["mean_before"].isna() & part["mean_after"].notna()
        part["sd_newly_available"] = part["sd_before"].isna() & part["sd_after"].notna()
        part["changed"] = (
            part["n_change"].ne(0)
            | ~np.isclose(part["mean_before"], part["mean_after"], equal_nan=True)
            | ~np.isclose(part["sd_before"], part["sd_after"], equal_nan=True)
        )
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

station_keys = ["station", "index", "season", "phase", "category"]
station_summary_comparison = summary_comparison(
    original_results["station_summary"], spliced_results["station_summary"], station_keys
)
ensemble_keys = ["index", "season", "phase", "category"]
ensemble_summary_comparison = summary_comparison(
    original_results["ensemble_summary"], spliced_results["ensemble_summary"], ensemble_keys
)
print("Changed station/index/season/category/metric rows:", int(station_summary_comparison["changed"].sum()))
print("Changed ensemble/index/season/category/metric rows:", int(ensemble_summary_comparison["changed"].sum()))
display(ensemble_summary_comparison.loc[ensemble_summary_comparison["changed"]])